<a href="https://colab.research.google.com/github/ovifernandez/pruebaopengeoai/blob/develop/model-trainer-kaggle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
# 1. Instalamos geoai (al ser muy ligero, dejamos que lo baje de internet)
%pip install geoai-py -q


In [ ]:
import os
import geoai

ModuleNotFoundError: No module named 'geoai'

In [ ]:
import os

input_folder = "/content/drive/MyDrive/AGRIA/input/parcelas/B7"
out_folder = "/content/drive/MyDrive/AGRIA/maskrcnn/mask_10epochs"
os.makedirs(out_folder, exist_ok=True)

Entrenamos el modelo Mask R-CNN sobre nuestros tiles generados, para que clasifique, localice bboxes y aplique máscaras a cada cepa detectada.

In [ ]:
geoai.train_instance_segmentation_model(
    images_dir=f"{input_folder}/images",
    labels_dir=f"{input_folder}/labels",
    output_dir=f"{out_folder}",
    num_classes=2,  # clase fondo y clase cepa. En un futuro, se añadirá clase tronco
    num_channels=5, # 3 para imágenes RGB, 5 para imágenes MSP
    batch_size=4, # Para no consumir excesiva VRAM, y no provocar un error de Out of Memory a mitad de ejecución.
    num_epochs=30, # 10 para una PoC, 50 para entrenamiento real con dataset augmentado.
    learning_rate=0.0005, # Learning rate menos agresivo que el de por defecto, para un descenso de gradiente suave y controlado con un batch sizze de 4.
    val_split=0.2,
    visualize=False,
    verbose=True,
)

In [ ]:
modelo_entrenado = f"/kaggle/working/best_model.pth"
ruta_prediccion = "/kaggle/working/prediccion_cepas.tif"


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

print("Iniciando escaneo absoluto del directorio de trabajo...")
archivos_encontrados = 0

# Buscamos en todo /kaggle/working sin importar cuántas subcarpetas haya
for raiz, directorios, archivos in os.walk('/kaggle/working'):
    for archivo in archivos:
        if archivo.endswith(('.pth', '.pt', '.weights')):
            ruta_completa = os.path.join(raiz, archivo)
            peso_mb = os.path.getsize(ruta_completa) / (1024 * 1024)
            print(f"📍 ¡LOCALIZADO! -> {ruta_completa} ({peso_mb:.2f} MB)")
            archivos_encontrados += 1

if archivos_encontrados == 0:
    print("❌ Negativo. La librería no ha guardado nada tras 1 época.")